# Prepare expression for metaprogram scoring

Combine predicted expression from melanoma-enriched spots into one matrix.


## Libraries


In [ ]:
from pathlib import Path
import json
import gzip
import pandas as pd
import numpy as np


## Settings

Use the same `config.json` as notebook 01. Set `use_generated = True` to use its new predictions, or leave it as `False` to use the existing prediction files.


In [ ]:
root = Path.cwd()
config_path = root / 'config.json'
use_generated = False


## Load the slide metadata and selected spots


In [ ]:
c = json.loads(config_path.read_text())
w = Path(c['workspace']) / 'virtualHE'
out = Path(c['output']) / 'virtualHE' / 'data'
out.mkdir(parents=True, exist_ok=True)
source = w / 'outputs/he_modulescore_existing/inputs_shared_mask'
metadata = pd.read_csv(source / 'slides.csv')
historical = pd.read_csv(source / 'pretrained_spots.csv.gz')
schema = json.loads((source / 'schema.json').read_text())
genes = json.loads((root / 'inputs/panel.json').read_text())['supported_genes']
cohort = pd.read_csv(w / 'inputs/cohort.csv')
new = [s for s in cohort.slide if s not in set(metadata.slide)]
records = []


## Load expression for control-gene selection


In [ ]:
reference_dir = w / 'outputs/xenium_background_check_20261021/inputs'
reference_spots = pd.read_csv(reference_dir / 'spots.csv.gz')
reference_schema = json.loads((reference_dir / 'schema.json').read_text())
with gzip.open(reference_dir / 'expression.bin.gz', 'rb') as stream:
    reference_expression = np.frombuffer(stream.read(), dtype='<f4').reshape(-1, len(genes))
assert reference_schema['genes'] == genes


## Combine expression across slides

Keep the gene order and selected melanoma spots consistent across prediction files.


In [ ]:
with gzip.open(out / 'expression.bin.gz', 'wb') as stream:
    for slide in list(metadata.slide) + new:
        if slide in new:
            prediction = w / f'outputs/he_gene_cohort/{slide}_gene_predictions.npz'
            with np.load(w / f'outputs/historical_extension_20261019/{slide}_historical_mask.npz') as z:
                selected = np.flatnonzero(z['melanoma_mask'])
        else:
            prediction = w / metadata.set_index('slide').loc[slide, 'prediction_path']
            selected = historical.loc[historical.slide.eq(slide), 'spot_index'].to_numpy()
        if use_generated and slide in set(cohort.slide):
            with np.load(Path(c['output']) / 'virtualHE' / 'predictions' / f'{slide}.npz') as a:
                order = a['genes'].astype(str).tolist()
                expression = a['pretrained'][selected][:, [order.index(g) for g in genes]]
                xy = a['centers_xy'][selected]
                # Keep the same melanoma spots when using new expression predictions.
                keep = np.zeros(len(a['centers_xy']), dtype=bool)
                keep[selected] = True
                print(slide, 'regenerated mask differences:', int(np.sum(keep != a['melanoma_mask'])))
        elif not prediction.exists() and slide not in set(cohort.slide):
            rows = np.flatnonzero(reference_spots.slide.eq(slide).to_numpy())
            assert np.array_equal(reference_spots.iloc[rows].spot_index.to_numpy(), selected)
            expression = reference_expression[rows]
            xy = reference_spots.iloc[rows][['x', 'y']].to_numpy()
            print(slide, 'using retained control-pool expression')
        else:
            with np.load(prediction) as a, np.load(w / f'outputs/xenium_panel_decoder/{slide}_additional_genes.npz') as b:
                assert np.array_equal(a['centers_xy'], b['centers_xy'])
                order = a['genes'].astype(str).tolist() + b['genes'].astype(str).tolist()
                expression = np.concatenate([a['pretrained'][selected], b['pretrained'][selected]], axis=1)[:, [order.index(g) for g in genes]]
                xy = a['centers_xy'][selected]
        stream.write(expression.astype('<f4').tobytes())
        records.append(pd.DataFrame(dict(slide=slide, spot_index=selected, x=xy[:, 0], y=xy[:, 1], original_pool=slide not in new)))


## Save expression and spot metadata


In [ ]:
spots = pd.concat(records, ignore_index=True)
assert spots.original_pool.sum() == 104840
spots.to_csv(out / 'spots.csv.gz', index=False)
(out / 'schema.json').write_text(json.dumps(dict(genes=genes, signatures=schema['signatures'], n_fit=104840, n_spots=len(spots)), indent=2))
print(len(spots), 'spots prepared')
